# Chapter 1 &mdash; A Tour of Jove's Machines

**Concept 6 of the Chapter 1 decomposition:** *A Tour of Jove's Machines*

You have just seen what a machine <i>is</i>. Here is what one looks like when you can run it &mdash; the full animation panel, and a DFA built, drawn, run and stepped.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Tour-of-Jove-Machines/Concept-Tour-of-Jove-Machines.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Concept 5 defined a Turing machine on paper. The natural next question is what one of
these looks like when you can **actually run it** &mdash; and that is the whole of this
notebook. It is not a section of the book; it is a tour of the tooling, put here
because this is where you first need it.

Two ways in, and you should try both.

**The full animation panel** comes first. Tabs for *Edit*, *Animate* and *Help*,
arriving with ready-made machines. You can see a machine and step it **before you have
learned any syntax at all**. It was written by Paul C.J. Carlson, who took this class
and then TA'd it.

**Then the slow way**, which is the workflow the rest of the book uses:

* write the machine as **markdown** and convert it with `md2mc`,
* **draw** it with `dotObj_dfa`,
* **run** it with `accepts_dfa`,
* **step** it with `AnimateDFA`.

Four functions. That is genuinely most of what you need to follow the book.

## 2. Definitions

### The full animation panel

Run the cell. You get *Edit*, *Animate* and *Help* tabs, pre-loaded with machines.
Edit one, switch to *Animate*, type an input string and press play &mdash; there is no
syntax to learn first. *Options* at the bottom controls colours, edge fusing and stack
depth.

The *Help* tab is worth reading on its own: it documents the markdown for all four
machine types.

In [ ]:
from jove.JoveEditor import *
JoveEditor(examples=True)

### Now the slow way: a machine as markdown

Machines are written in **Knuth's literate style** &mdash; the design reasoning lives
in the comments, beside the transitions it explains.

Two naming rules do all the work: a state whose name starts with `I` is **initial**,
one starting with `F` is **final**, and `IF` is both. Everything after `!!` is a
comment.

This one accepts strings containing `010`. The comments are deliberately over-done,
this being your first machine.

In [ ]:
DFA010 = md2mc('''
DFA

!! Name states to reflect the information being recorded.

!! The initial state is not final -- no 010 has been seen yet -- so it is
!! called "I" and not "IF".

I   : 1 -> I    !! A '1' makes no progress toward 010; throw it away
I   : 0 -> S0   !! A '0' is progress; record that in the state name

S0  : 0 -> S0   !! No further progress, but nothing lost either; stay
S0  : 1 -> S01  !! Progress toward 010

S01 : 1 -> I    !! A spoiler; back to the start
S01 : 0 -> F    !! Seen 010.  No more work to do.

F   : 0|1 -> F  !! Stay accepting, whatever follows
''')

print('states :', sorted(DFA010['Q']))
print('start  :', DFA010['q0'], '   final:', DFA010['F'])

### Draw it

`FuseEdges=True` collapses parallel edges into one with the labels stacked &mdash;
almost always what you want to look at.

In [ ]:
dotObj_dfa(DFA010, FuseEdges=True)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;5.&nbsp;Defining a Computer: the Turing Machine](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Turing-Machine-Definition/Concept-Turing-Machine-Definition.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1-Intro/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;7.&nbsp;Convergence of Models, and the Church–Turing Thesis](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Convergence-Church-Turing/Concept-Convergence-Church-Turing.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Run it.** `accepts_dfa` takes the machine and a string.

In [ ]:
for s in ['010', '0010', '11011', '', '1', '0110010', '111']:
    print('   %-9r contains 010 ?  %s' % (s, accepts_dfa(DFA010, s)))

**Test it in bulk, in numeric order.** `nthnumeric(i, ['0','1'])` enumerates
$\Sigma^*$ in *numeric order* &mdash; by length first, then lexicographically &mdash;
so a prefix of that list is a complete test set for all short strings, with nothing
skipped and nothing repeated.

In [ ]:
tests = [nthnumeric(i, ['0', '1']) for i in range(32)]
print('the first 32 strings over {0,1}, in numeric order:')
print('  ', tests)

good = list(filter(lambda x: accepts_dfa(DFA010, x), tests))
print()
print('of those, the ones containing 010:')
print('  ', good)

assert all('010' in s for s in good)
assert all('010' not in s for s in tests if s not in good)
print()
print('Checked both ways: everything accepted contains 010, and nothing')
print('rejected does.  That is a test, not a demonstration.')

### Where to find what

What you just used is the whole toolkit; the rest is the same four moves on other
machine types.

| You want to | Use | Lives in |
|---|---|---|
| turn markdown into a machine | `md2mc` | `jove.Def_md2mc` |
| draw one | `dotObj_dfa`, `dotObj_nfa`, `dotObj_pda`, `dotObj_tm` | `jove.DotBashers` |
| run one | `accepts_dfa`, `run_nfa`, `run_pda`, `explore_tm` | `jove.Def_DFA` &hellip; `jove.Def_TM` |
| step one | `AnimateDFA`, `AnimateNFA`, `AnimatePDA`, `AnimateTM` | `jove.Animate*` |
| all four at once | `JoveEditor` | `jove.JoveEditor` |
| operations on languages | `lcat`, `lstar`, `lunion`, `nthnumeric`, &hellip; | `jove.LangDef` |

And **chapter by chapter**: each folder in this repository holds one notebook per
concept, in reading order. The folder name carries the topic, so you can go
straight to it &mdash; and the search box above takes the tag too, so typing
`BDD` or `PDA` finds the chapter.

| Folder | Folder | Folder | Folder |
|---|---|---|---|
| `Chapter1-Intro` what machines think | `Chapter6-DFAOps` operations on DFA | `Chapter11-CFG` context-free languages | `Chapter16-NPC` NP-completeness |
| `Chapter2-Lang` defining languages | `Chapter7-NFA` NFA | `Chapter12-PDA` pushdown automata | `Chapter17-BDD` BDDs as minimal DFA |
| `Chapter3-Star` Kleene star | `Chapter8-RE` regular expressions | `Chapter13-TM` Turing machines | `Chapter18-Lambda` computability with lambdas |
| `Chapter4-DFA` basics of DFA | `Chapter9-NFA2RE` NFA to RE | `Chapter14-Interp` interplay of languages | `Basics` discrete math (App. A) |
| `Chapter5-DFADsg` designing DFA | `Chapter10-Deriv` derivative-based matching | `Chapter15-PCP` Post correspondence | |

Every chapter folder has a `README.md` listing its concepts, and the link strip in the
middle of each notebook walks you to the previous and the next one.

### And now, please read the book

This repository is the *laboratory*. The argument is in the book &mdash; **Automata
and Computability: A Programmer's Perspective** &mdash; and the notebooks are worth
very little without it. Running a machine shows you *that* something happens; only the
book tells you *why it must*.

So **follow the side-bars.** They exist to send you here at the moment a construction
is worth running rather than reading, and to send you back once you have run it. Read
the chapter, work the notebook it points at, and you get something neither half gives
on its own.

## 4. Animation

Step the machine you just built. Type an input and press play, or walk
it one symbol at a time with the arrows &mdash; the current state lights up as each
symbol is consumed.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(DFA010, FuseEdges=True)

## 5. Exercises


1. In the panel at the top, switch the *Edit* tab to **PDA** and animate the example
   it comes with. What does the stack display show that a DFA animation has no room
   for?
2. Change `DFA010` to accept strings containing `0110` instead. Draw it, and check it
   with the `nthnumeric` test above rather than by eye.
3. `dotObj_dfa(DFA010, FuseEdges=False)` draws every edge separately. Compare the two
   pictures. On what kind of machine would fusing hide something you wanted to see?
4. The test above enumerates 32 strings, and the last one printed is `'00000'` --
   so 32 is *not* the count of strings of length $\le 4$. Work out what that count
   actually is, and say which `range(n)` covers exactly the strings of length
   $\le 5$ and no more.
5. Take one machine from the panel's *Edit* tab, write it out yourself as `md2mc`
   markdown in a cell of your own, and confirm the two agree on ten inputs.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1-Intro/Concept-Tour-of-Jove-Machines')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')